# Track 7 — DataOps, FinOps y Salud
**Rol:** CRB_INFRAESTRUCTURA | **Tiempo:** 15 min | **Criterio:** Observabilidad, costos, atribución, mejora continua

In [ ]:
USE ROLE CRB_INFRAESTRUCTURA;
USE DATABASE CREDIBANCO_HOL;
USE WAREHOUSE CREDIBANCO_HOL_WH;

## Bloque 1 — Evidencia: Observabilidad nativa

In [ ]:
-- Consumo por warehouse (últimos 30 días)
SELECT WAREHOUSE_NAME, SUM(CREDITS_USED) AS total_credits,
       COUNT(*) AS num_queries
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY
WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
GROUP BY 1 ORDER BY 2 DESC;

In [ ]:
-- Top 10 queries más costosas
SELECT QUERY_ID, LEFT(QUERY_TEXT, 80) AS preview,
       WAREHOUSE_NAME, ROUND(TOTAL_ELAPSED_TIME/1000,1) AS seconds
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE START_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
ORDER BY TOTAL_ELAPSED_TIME DESC LIMIT 10;

In [ ]:
-- Tags de atribución: chargeback por dominio
SELECT * FROM TABLE(CREDIBANCO_HOL.INFORMATION_SCHEMA.TAG_REFERENCES(
  'CREDIBANCO_HOL_WH', 'WAREHOUSE'
));

## Bloque 2 — Ejecutar: Alerta + Tag chargeback

In [ ]:
-- Crear alerta de consumo elevado
CREATE OR REPLACE ALERT CREDIBANCO_HOL.PLATAFORMA.ALERTA_CONSUMO_<TU_USUARIO>
  WAREHOUSE = CREDIBANCO_HOL_WH
  SCHEDULE = '60 MINUTE'
  IF (EXISTS (
    SELECT 1 FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY
    WHERE START_TIME >= DATEADD('hour', -1, CURRENT_TIMESTAMP()) AND CREDITS_USED > 5
  ))
  THEN CALL SYSTEM$LOG('info', 'Alerta: consumo elevado detectado');

In [ ]:
-- Tag de chargeback al warehouse
ALTER WAREHOUSE CREDIBANCO_HOL_WH
  SET TAG CREDIBANCO_HOL.GOBIERNO.TAG_DOMINIO = 'PLATAFORMA';

## Bloque 3 — CoCo
Copia este prompt en Cortex Code:

> **Analiza el consumo de los últimos 7 días de la cuenta, identifica el warehouse más costoso, las queries más lentas, y genera un Streamlit dashboard FinOps con KPIs, gráficos de tendencia y tabla de top queries.**

In [ ]:
-- Verificación final
SELECT 'T7_COMPLETO' AS status;